# Load the data

In [1]:
import tarfile
import urllib.request
from pathlib import Path

import numpy as np


def extract_reviews(movie_reviews: list[Path]) -> list[str]:
    """Extract movie reviews from a tar file."""
    data = []
    for file in movie_reviews:
        with open(file, "r") as f:
            movie_review = f.readline()
        data.append(movie_review.strip())
    return data


def load_movie_reviews() -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    tarball_path = Path("datasets/aclImdb_v1.tar.gz")
    if not tarball_path.is_file():
        print("Downloading Stanford Large Movie Review Dataset...")
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
        urllib.request.urlretrieve(url, tarball_path)
        with tarfile.open(tarball_path) as movies_tarball:
            movies_tarball.extractall(path="datasets", filter="data")

    train_data = []
    train_pos_files = [f for f in Path("datasets/aclImdb/train/pos").iterdir() if f.is_file()]
    train_neg_files = [f for f in Path("datasets/aclImdb/train/neg").iterdir() if f.is_file()]
    train_data.extend(extract_reviews(train_pos_files))
    train_data.extend(extract_reviews(train_neg_files))
    train = np.array(train_data, dtype=object)
    train_labels = np.array([1] * len(train_pos_files) + [0] * len(train_neg_files))

    test_data = []
    test_pos_files = [f for f in Path("datasets/aclImdb/test/pos").iterdir() if f.is_file()]
    test_neg_files = [f for f in Path("datasets/aclImdb/test/neg").iterdir() if f.is_file()]
    test_data.extend(extract_reviews(test_pos_files))
    test_data.extend(extract_reviews(test_neg_files))
    test = np.array(test_data, dtype=object)
    test_labels = np.array([1] * len(test_pos_files) + [0] * len(test_neg_files))

    return train, test, train_labels, test_labels


In [2]:
train, test, train_labels, test_labels = load_movie_reviews()

In [3]:
train.shape

(25000,)

In [4]:
train_labels.shape

(25000,)

In [5]:
train[19]

"It may be the remake of 1987 Autumn's Tale after eleven years, as the director Mabel Cheung claimed. Mabel employs rock music as the medium in this movie to express her personal attitude to life, in which love, desire and the consequential frustration play significantly crucial roles. Rock music may not be the best vehicle to convey the profound sentiment, and yet it is not too inappropriate to utilize it as the life of underground rock musicians is bitterly more intense than an ordinary one. The director focuses on the depiction of subtle affection and ultimate vanity of life rather than mere rock music. The love between father and son, lovers, and friends is delicately and touchingly delivered through the fine performance. Mabel does not attempt to beautify rock musicians as artists at all, instead, she tries to reproduce a true life on screen, making huge efforts of years' working on this project and gathering information in Beijing underground pubs.<br /><br />Daniel has given pro

In [6]:
train_labels[19]

np.int64(1)

In [7]:
test.shape

(25000,)

In [8]:
test_labels.shape

(25000,)

# Shuffle the order of positive and negative sentiment

In [9]:
from sklearn.utils import shuffle

train, train_labels = shuffle(train, train_labels, random_state=42)
test, test_labels = shuffle(test, test_labels, random_state=42)

# Use a smaller subset of the training data for evaluating different models

In [10]:
from sklearn.model_selection import train_test_split

X_train = train[:10000].copy()
y_train = train_labels[:10000].copy()

X_train, X_test, y_train, y_test = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)

In [11]:
X_train.shape

(8000,)

# Create Preprocess Pipeline

In [12]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from preprocess import TextPreprocessingTransformer

preprocess_pipeline = Pipeline([
    ("text_preprocessing", TextPreprocessingTransformer()),
    ("vectorization", TfidfVectorizer(ngram_range=(1, 2), stop_words="english")),
])

X_train_transformed = preprocess_pipeline.fit_transform(X_train)

# Evaluate different models and select the best one

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

scores_lgr = cross_val_score(LogisticRegression(random_state=42), X_train_transformed, y_train, cv=3,
                             scoring="accuracy")
scores_lgr.mean()

np.float64(0.8443756514681726)

In [14]:
from sklearn.linear_model import SGDClassifier

scores_sgd = cross_val_score(SGDClassifier(random_state=42), X_train_transformed, y_train, cv=3, scoring="accuracy")
scores_sgd.mean()

np.float64(0.8686251521635565)

In [34]:
from sklearn.svm import SVC

scores_svc = cross_val_score(SVC(random_state=42), X_train_transformed, y_train, cv=3, scoring="accuracy")
scores_svc.mean()

np.float64(0.831875)

In [35]:
from sklearn.neighbors import KNeighborsClassifier

scores_knn = cross_val_score(KNeighborsClassifier(), X_train_transformed, y_train, cv=3, scoring="accuracy")
scores_knn.mean()

np.float64(0.7322916666666668)

In [36]:
from sklearn.ensemble import RandomForestClassifier

scores_forest = cross_val_score(RandomForestClassifier(random_state=42), X_train_transformed, y_train, cv=3,
                                scoring="accuracy")
scores_forest.mean()

np.float64(0.8183333333333334)

In [45]:
from sklearn.naive_bayes import MultinomialNB

scores_multi = cross_val_score(MultinomialNB(), X_train_transformed, y_train, cv=3, scoring="accuracy")
scores_multi.mean()

np.float64(0.8567495735951236)

# Hyperparameters tuning

I decided to go with `SGDClassifier` since it performs better than the rest in terms of accuracy and training time

In [18]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "loss": ["hinge", "log_loss", "modified_huber"],
    "penalty": ["l1", "l2"],
    "learning_rate": ["constant", "optimal"]
}

grid_search = GridSearchCV(SGDClassifier(random_state=42), param_grid, scoring="accuracy", cv=3, refit=True)
grid_search.fit(X_train_transformed, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",SGDClassifier(random_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': ['constant', 'optimal'], 'loss': ['hinge', 'log_loss', ...], 'penalty': ['l1', 'l2']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate i

In [19]:
grid_search.best_score_

np.float64(0.8686251521635565)

In [20]:
grid_search.best_params_

{'learning_rate': 'optimal', 'loss': 'hinge', 'penalty': 'l2'}

In [21]:
from sklearn.metrics import precision_score, accuracy_score, recall_score, f1_score

X_test_transformed = preprocess_pipeline.transform(X_test)
y_pred = grid_search.predict(X_test_transformed)

print("Accuracy score:", accuracy_score(y_test, y_pred))
print("Precision score:", precision_score(y_test, y_pred))
print("Recall score", recall_score(y_test, y_pred))
print("F1 score:", f1_score(y_test, y_pred))

Accuracy score: 0.8585
Precision score: 0.8410159924741298
Recall score 0.8869047619047619
F1 score: 0.8633510381458233


# Train the model on the full dataset

In [22]:
full_pipeline = Pipeline([
    ("preprocess", preprocess_pipeline),
    ("sgd", SGDClassifier(random_state=42, **grid_search.best_params_)),
])

X_train = train
X_test = test
y_train = train_labels
y_test = test_labels

full_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('sgd', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('text_preprocessing', ...), ('vectorization', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes i

In [23]:
from sklearn.metrics import precision_score, recall_score, accuracy_score

y_pred = full_pipeline.predict(X_test)

print("Accuracy score:", accuracy_score(y_test, y_pred))
print("Precision score:", precision_score(y_test, y_pred))
print("Recall score", recall_score(y_test, y_pred))
print("F1 score:", f1_score(y_test, y_pred))

Accuracy score: 0.87568
Precision score: 0.8705808080808081
Recall score 0.88256
F1 score: 0.8765294771968855


# Save the final model

In [18]:
import joblib

joblib.dump(full_pipeline, "models/movie_reviews.joblib", compress=3)

['models/movie_reviews.joblib']